# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

***Note: All record sets, fields, and columns are referenced by their `@id` fields, as specified by the Croissant schema.***

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s per the Croissant specification.

In [ ]:
# Display all available record sets by @id
record_set_ids = [rs.id for rs in metadata.record_sets]
print('Available record sets (@id):')
for rs in metadata.record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    # Show contained fields
    field_ids = [f.id for f in rs.fields]
    print(f"  Field @ids: {field_ids}\n")

Let's display one example record from each record set using the proper `@id` reference.

In [ ]:
# Show one record per record set (referenced by @id)
for rs in record_set_ids:
    print(f"Sample from record set {rs}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs)):
            print(rec)
            if i == 0:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print('-'*50)


## 3. Data Extraction
Load data from available record sets into DataFrames for further analysis. This demonstration loads all available record sets, referenced by their `@id`. Each field (column) is also referenced using its `@id`.

In [ ]:
# Load all data into DataFrames by record set @id
dataframes = dict()

for record_set in record_set_ids:
    print(f"Loading record set: {record_set}")
    try:
        # records() yields dicts with field @id as keys
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"  Fields (@id): {df.columns.tolist()}")
            print(df.head(2))
        else:
            print("  (No records)")
    except Exception as e:
        print(f"  Could not load: {e}")
print(f"\nLoaded DataFrames (record_set @id): {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key categorical attributes. All columns are referenced using their `@id` as per the Croissant schema.

In [ ]:
# Example: Analyze fields in the main data record set (likely the protein results set)
# We'll look for a record set containing quantitative fields. List all numeric-type columns.

# Find a dataframe with numeric columns
main_rs_id = None
numeric_cols = []
for rs_id, df in dataframes.items():
    # Pick column candidates (float or int)
    cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if cols and len(df) > 5:
        main_rs_id = rs_id
        numeric_cols = cols
        break
if main_rs_id is None:
    raise RuntimeError("Could not locate a main record set with numeric fields.")

print(f"Analyzing main record set: {main_rs_id}")
print(f"Available numeric columns (@id): {numeric_cols}")

# Choose one numeric field (@id) for demonstration:
numeric_field_id = numeric_cols[0]
df = dataframes[main_rs_id]

threshold = df[numeric_field_id].quantile(0.7)  # 70th percentile as example threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (top 30%): {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If available, pick a categorical field (@id) for grouping
categorical_cols = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
if categorical_cols:
    group_field_id = categorical_cols[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by categorical field {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    group_field_id = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This example shows histograms for the selected numeric field and a boxplot per group for a categorical field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f'{numeric_field_id} by group ({group_field_id})')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry Analysis dataset via its Croissant schema, inspected its record sets and fields using their `@id`, extracted data into DataFrames, and performed standard EDA tasks such as filtering and normalization. All entities and field references strictly followed their Croissant `@id` for reproducibility and schema alignment.

Further steps might include domain-specific statistical analysis, integration with protein databases, or advanced visualization for mass spec data exploration.